In [1]:
import pandas as pd
import numpy as np
import ast, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise         import cosine_similarity
from sklearn.model_selection          import train_test_split, cross_val_score, KFold
from sklearn.metrics                  import mean_squared_error, r2_score
from sklearn.preprocessing            import LabelEncoder, OrdinalEncoder
import xgboost as xgb

print("All libraries loaded!")

All libraries loaded!


In [2]:
DATA_FOLDER = r"Dataset"

profiles = pd.read_csv(f"{DATA_FOLDER}/profiles.csv")
pairs    = pd.read_csv(f"{DATA_FOLDER}/compatibility_pairs.csv")

print(f"Profiles : {profiles.shape[0]:,} rows")
print(f"Pairs    : {pairs.shape[0]:,} rows")

Profiles : 50,000 rows
Pairs    : 4,999,890 rows


In [3]:
from pathlib import Path

# create demo folder
demo_folder = Path(DATA_FOLDER) / "demo"
demo_folder.mkdir(parents=True, exist_ok=True)

# save top 15 rows from both CSVs
profiles_demo = profiles.head(15)
pairs_demo    = pairs.head(15)

profiles_demo_path = demo_folder / "profiles_top15.csv"
pairs_demo_path    = demo_folder / "compatibility_pairs_top15.csv"

profiles_demo.to_csv(profiles_demo_path, index=False)
pairs_demo.to_csv(pairs_demo_path, index=False)

print("Demo files created:")
print(profiles_demo_path)
print(pairs_demo_path)

# optional preview
display(profiles_demo)
display(pairs_demo)

Demo files created:
Dataset\demo\profiles_top15.csv
Dataset\demo\compatibility_pairs_top15.csv


,profile_id,name,email,location,headline,about,current_role,current_company,industry,years_experience,seniority_level,skills,experience,education,connections,goals,needs,can_offer,remote_preference,source
0,ab04b973af478550ddf247879393df42,Daniel Doyle,garzaanthonyexample.org,"East William, AK",Analyst Product Building impactful solutions,Experienced professional focused on driving gr...,Assistant,Microsoft,Healthcare,2,entry,"['Prototyping', 'Go', 'C', 'C', 'NLP']","[{'title': 'Assistant', 'company': 'Google', '...","[{'school': 'Penn', 'degree': 'MS', 'field': '...",106,"['Get promoted', 'Build network']","['funding', 'mentorship', 'business advice']","['partnership opportunities', 'investment', 'c...",remote,synthetic
1,b620e3fa2ec361b1d728115eeabb71af,Jennifer Cole,lisa02example.net,"Petersonberg, IL",Senior Engineer Design Building impactful so...,Passionate about building innovative solutions...,Lead Data Scientist,Stripe,Consulting,9,senior,"['Business Development', 'SQL', 'Ruby', 'Marke...","[{'title': 'Senior Engineer', 'company': 'Rapi...","[{'school': 'Princeton', 'degree': 'MBA', 'fie...",2372,"['Strategic role', 'Scale impact']","['job opportunities', 'mentorship', 'clients']","['industry connections', 'consulting', 'produc...",hybrid,synthetic
2,cfeeb31581a0b3e0515c01691b9dc2b5,Brent Abbott,lindsay78example.org,"Millerport, MP",Software Engineer Product Building impactful...,Passionate about building innovative solutions...,Data Scientist,NextGen,Telecommunications,5,mid,"['NLP', 'Business Development', 'Sales', 'Big ...","[{'title': 'Consultant', 'company': 'ScaleUp',...","[{'school': 'CMU', 'degree': 'MS', 'field': 'D...",874,"['Specialize', 'Mentor others']","['job opportunities', 'business advice', 'care...","['partnership opportunities', 'product feedbac...",onsite,synthetic
3,5d54826665a5898662661a96719cc4a7,Corey Jones,kendragallowayexample.org,"South Joshuastad, GA",Chief Data Officer Tech Building impactful s...,Passionate about building innovative solutions...,COO,NextGen,Healthcare,17,executive,"['Sketch', 'Machine Learning', 'AWS', 'TensorF...","[{'title': 'VP Engineering', 'company': 'Tesla...","[{'school': 'MIT', 'degree': 'BS', 'field': 'E...",3259,"['Build company', 'Advisory roles']","['clients', 'hiring', 'career guidance']","['consulting', 'career advice', 'hiring referr...",onsite,synthetic
4,6ad3c64c6cb4bac60b692f3d5bab271d,Timothy Wong,amandasanchezexample.com,"Nelsonside, IN",Consultant Design Building impactful solutions,Strategic thinker with expertise in scaling or...,Product Manager,NextGen,Healthcare,4,mid,"['Data Science', 'DevOps', 'Flask', 'Analytics...","[{'title': 'Software Engineer', 'company': 'Ai...","[{'school': 'Berkeley', 'degree': 'MS', 'field...",372,"['Specialize', 'Lead projects']","['clients', 'hiring', 'funding']","['partnership opportunities', 'career advice',...",remote,synthetic
5,38b9dbc8ee3fdc67c12e55fb13353a02,Crystal Johnson,richard13example.net,"Jasonfort, MO",Senior Product Manager Business Building imp...,Experienced professional focused on driving gr...,Senior Designer,Meta,Transportation,15,senior,"['Machine Learning', 'TypeScript', 'Finance', ...","[{'title': 'Senior Engineer', 'company': 'Scal...","[{'school': 'Duke', 'degree': 'MS', 'field': '...",1832,"['Strategic role', 'Executive position']","['clients', 'business advice', 'funding']","['industry connections', 'hiring referrals', '...",hybrid,synthetic
6,81ecf3a9f4073708e3107805d8d37182,Charles Mcgee,julie69example.com,"Jeffreyberg, NY",Lead Data Scientist Design Building impactfu...,Strategic thinker with expertise in scaling or...,Senior Product Manager,Amazon,Energy,15,senior,"['NLP', 'Operations', 'C', 'Spark', 'UIUX Desi...","[{'title': 'Senior Product Manager', 'company'...","[{'school': 'UCLA', 'degree': 'MBA', 'field': ...",1647,"['Scale impact', 'Executive position']","['network connections', 'hiring', 'career guid...","['speaking opportunities', 'product feedback',...",hybrid,synthetic
7,253

,skill_match_score,skill_complementarity_score,network_value_a_to_b,network_value_b_to_a,career_alignment_score,experience_gap,industry_match,geographic_score,seniority_match,compatibility_score,mutual_benefit_explanation,pair_id,profile_a_id,profile_b_id
0,0.000000,0.0,5.3,14.55,80.0,0,0.0,60.0,85.0,24.98,Peer-level relationship - can learn together,742f902f23b9d1be5fa0ba0816e3490b,ab04b973af478550ddf247879393df42,fdf3243d1ad97255e0ce313aebc0be79
1,5.555556,0.0,5.3,42.80,80.0,2,0.0,60.0,100.0,30.33,Peer-level relationship - can learn together,fc6c6a8029bc4e9beb5dd8186147f042,ab04b973af478550ddf247879393df42,371bc2adbdc4ca8f0dd16d373f85f2ae
2,5.555556,0.0,5.3,38.80,80.0,2,0.0,60.0,100.0,29.73,Peer-level relationship - can learn together,6cb0929495e49ddd5dff151ead7b3f5e,ab04b973af478550ddf247879393df42,d18e7cd91fc4e621fd6879ca5ef6e1b2
3,8.000000,0.0,5.3,80.00,40.0,27,0.0,60.0,50.0,28.39,Valuable network connections in same industry,7b191de3be4c52fd0874f31936d51819,ab04b973af478550ddf247879393df42,6615dabdd3b5b8f9627ca933dc3d9ae3
4,7.142857,0.0,5.3,28.60,90.0,5,0.0,60.0,100.0,30.51,Ideal mentorship gap (5 years experience diffe...,7b568a4ee7254a9b388e6430ac252568,ab04b973af478550ddf247879393df42,fc73d0e790dea954d20db176f638ab86
5,0.000000,0.0,5.3,14.30,80.0,0,0.0,60.0,85.0,24.94,Peer-level relationship - can learn together,5457d21562aa1073945497320563d7d5,ab04b973af478550ddf247879393df42,cc4bccaddb71b25f4a1a21802eda548a
6,4.347826,0.0,5.3,80.00,40.0,23,0.0,75.0,50.0,29.16,Valuable network connections in same industry,4bb4cd1996fcffb39909f0587c56352a,ab04b973af478550ddf247879393df42,ec6229666265202f874cb816b00eba1a
7,5.000000,0.0,5.3,70.00,60.0,13,0.0,60.0,70.0,30.30,Valuable network connections in same industry,1285f0ac7e88492de686822a10977c73,ab04b973af478550ddf247879393df42,426c4308c4f63bb56802a73a8597fac7
8,8.333333,0.0,5.3,9.20,80.0,1,0.0,60.0,85.0,25.84,Peer-level relationship - can learn together,0a44ae167bbbced3771e35d0e7dbd688,ab04b973af478550ddf247879393df42,7b155c4fd08693354c19a88f0d4ad7c5
9,0.000000,0.0,35.3,39.55,80.0,0,100.0,60.0,85.0,33.23,Peer-level relationship - can learn together |...,f53a2d26f006b0471533d150722adeb0,ab04b973af478550ddf247879393df42,585e2b716e968cc16a47687b20dab15c
